# 01 · 학습 — Eye / Yawn CNN

이 노트북은 **직접 학습하는 CNN 두 개만** 다룹니다.
얼굴 검출기(MTCNN / YuNet)와는 **아무 관계가 없습니다** — 검출기는 학습하지 않는
사전학습 모델이라 추론 노트북에서만 씁니다.

| | 역할 | 학습 |
|---|---|---|
| **Eye CNN** | 눈 감김 분류 (Closed / Open) | **여기서 학습** |
| **Yawn CNN** | 하품 분류 (yawn / no_yawn) | **여기서 학습** |
| 얼굴 검출기 | 얼굴 박스 + 랜드마크 5점 | 안 함 (사전학습) |

## 산출물

```
model/artifacts/
    eye_model.keras       가중치 14.8MB + Adam 옵티마이저 상태 29.6MB
    eye_history.json      학습 곡선 기록
    yawn_model.keras
    yawn_history.json
```

**한 번만 실행하면 됩니다.** 이후에는 `02_INFER_YuNet.ipynb` 만 쓰면 됩니다.
`.keras` 파일이 이미 있으면 이 노트북도 학습을 건너뜁니다
(다시 학습하려면 `FORCE_RETRAIN = True`).

CPU 실측 에폭당 약 8초, 모델 하나당 10에폭 ≈ 1.5분입니다.

> 커널: `C:\Users\psh03\AppData\Local\Programs\Python\Python312\python.exe`

이 노트북은 **Run All 로 끝까지 돌아갑니다.** (웹캠·무한루프 셀 없음)

## 1. 환경 점검

In [ ]:
import sys, platform

print("Python  :", sys.version.split()[0])
print("실행파일:", sys.executable)
print("OS      :", platform.platform())

import numpy as np
import tensorflow as tf
import cv2
import matplotlib

print("\nTensorFlow :", tf.__version__)
print("Keras      :", tf.keras.__version__)
print("OpenCV     :", cv2.__version__)
print("NumPy      :", np.__version__)
print("Matplotlib :", matplotlib.__version__)

# YuNet 은 OpenCV 내장이라 별도 패키지가 필요 없다
print("\nFaceDetectorYN (YuNet):", "OK" if hasattr(cv2, "FaceDetectorYN") else "없음 (OpenCV 4.5.4+ 필요)")

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "없음 (CPU 사용)")

if not gpus:
    print(
        "\n[안내] Windows 네이티브 TensorFlow 2.11+ 는 GPU를 지원하지 않습니다.\n"
        "       CPU로 학습합니다. 실측 에폭당 약 8초라 이 규모에서는 문제없습니다."
    )

## 2. 경로 설정

In [ ]:
import sys
from pathlib import Path

# =====================================================================
# 경로는 저장소 최상단 config.py 에서 가져온다. (README §10)
#
# 노트북에는 __file__ 이 없어서 config.py 의 "위치"만 cwd 기준으로 찾는다.
# 하지만 그 뒤의 모든 경로는 config.py 가 자신의 __file__ 로 계산하므로,
# 노트북을 어느 폴더에서 열든 결과가 같다. 못 찾으면 조용히 넘어가지 않고
# 즉시 멈춘다.
# =====================================================================
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "config.py").exists()), None)
if _root is None:
    raise FileNotFoundError(
        f"config.py 를 찾지 못했습니다 (탐색 시작: {_here}).\n"
        "저장소를 clone 한 폴더 안에서 노트북을 열었는지 확인하세요."
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import config

PROJECT_ROOT = config.PROJECT_ROOT

# 데이터셋은 git 추적 대상이 아니다. 없으면 다음 셀에서 kagglehub 로 받는다.
DATA_DIR  = config.DATA_DIR / "raw" / "dataset_new"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

# 학습 산출물 저장 위치. config 가 import 시점에 폴더를 만들어 둔다.
ARTIFACT_DIR = config.ARTIFACT_DIR

config.describe()
print()
print("DATA_DIR     :", DATA_DIR.relative_to(PROJECT_ROOT), " 존재:", DATA_DIR.exists())


## 3. 데이터셋 확인

In [ ]:
import glob, os

if not TRAIN_DIR.exists():
    print("로컬 데이터셋이 없습니다. kagglehub 로 다운로드합니다...")
    import kagglehub

    downloaded = kagglehub.dataset_download("serenaraju/yawn-eye-dataset-new")
    found = glob.glob(os.path.join(downloaded, "**", "dataset_new"), recursive=True)

    if not found:
        raise FileNotFoundError(f"dataset_new 를 찾지 못했습니다: {downloaded}")

    DATA_DIR  = Path(found[0])
    TRAIN_DIR = DATA_DIR / "train"
    TEST_DIR  = DATA_DIR / "test"
    print("다운로드 완료:", DATA_DIR)
else:
    print("로컬 데이터셋 사용:", DATA_DIR)

print("\nTRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)
print("\nTrain classes:", sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir()))
print("Test  classes:", sorted(p.name for p in TEST_DIR.iterdir() if p.is_dir()))

In [ ]:
# 클래스별 이미지 개수 (Colab 판의 !find 대체)
for split_dir in [TRAIN_DIR, TEST_DIR]:
    print("\n", split_dir)

    for cls in ["Closed", "Open", "no_yawn", "yawn"]:
        files = list((split_dir / cls).glob("*"))
        print(f"{cls:10s}: {len(files)} images")

### 데이터 특성

두 과제의 이미지 성격이 다릅니다. 추론 시 하품 입력 방식과 직결되니 확인하세요.

In [ ]:
import cv2

for cls in ["Closed", "Open", "yawn", "no_yawn"]:
    files = sorted((TRAIN_DIR / cls).glob("*"))[:3]
    for f in files:
        im = cv2.imread(str(f))
        shape = im.shape if im is not None else "READ FAIL"
        print(f"{cls:8s} {str(shape):18s} {f.name}")
    print()

print("정리:")
print("  Closed / Open   : 85~300px 눈 crop 이미지")
print("  yawn / no_yawn  : 640x480 얼굴 전체 이미지")

## 4. 학습 설정

`ModelCheckpoint(monitor="val_accuracy", save_best_only=True)` 가 매 에폭 검증 정확도를
확인해 **최고 기록일 때만** 덮어씁니다. 마지막 에폭이 아니라 가장 좋았던 시점이 남습니다.

- 다시 학습 → `FORCE_RETRAIN = True`
- 완전 초기화 → `model/artifacts/*.keras` 삭제

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

%matplotlib inline

# 재현성
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

IMG_SIZE   = (256, 256)
BATCH_SIZE = 32
EPOCHS     = 10          # 원본과 동일

EYE_MODEL_PATH  = ARTIFACT_DIR / "eye_model.keras"
YAWN_MODEL_PATH = ARTIFACT_DIR / "yawn_model.keras"

# 학습 곡선을 나중에 다시 그리려면 history 도 함께 저장해야 한다
EYE_HISTORY_PATH  = ARTIFACT_DIR / "eye_history.json"
YAWN_HISTORY_PATH = ARTIFACT_DIR / "yawn_history.json"

# =====================================================================
# 학습은 한 번만.  가중치 파일이 있으면 재학습하지 않고 불러온다.
#   다시 학습하고 싶으면 FORCE_RETRAIN = True 로 바꾸거나
#   model/artifacts/*.keras 를 지우세요.
# =====================================================================
FORCE_RETRAIN = False


class LoadedHistory:
    """저장된 history 를 Keras History 처럼 쓰기 위한 얇은 래퍼."""
    def __init__(self, history):
        self.history = history


def save_history(history, path):
    """History.history 를 JSON 으로 저장."""
    data = {k: [float(x) for x in v] for k, v in history.history.items()}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1)
    print("history 저장:", path)


def load_history(path):
    """저장된 history 를 불러온다. 없으면 None."""
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return LoadedHistory(json.load(f))


def needs_training(model_path):
    return FORCE_RETRAIN or not Path(model_path).exists()


print("IMG_SIZE  :", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS    :", EPOCHS)
print("FORCE_RETRAIN:", FORCE_RETRAIN)
print()
for name, p in [("eye ", EYE_MODEL_PATH), ("yawn", YAWN_MODEL_PATH)]:
    if p.exists():
        mb = p.stat().st_size / 1024 / 1024
        print(f"  {name} -> {p.name}  ({mb:.1f} MB)  이미 있음 -> 학습 건너뜀")
    else:
        print(f"  {name} -> {p.name}  없음 -> 학습함")

## 5. Eye 모델

In [ ]:
eye_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=["Closed", "Open"],
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

eye_val = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=["Closed", "Open"],
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# class 0 = Closed, class 1 = Open
print("\nclass_names:", eye_train.class_names)

In [ ]:
def build_eye_model():

    model = tf.keras.Sequential([

        tf.keras.layers.Input(shape=(256, 256, 3)),

        tf.keras.layers.Rescaling(1./255),

        tf.keras.layers.RandomFlip("horizontal"),

        tf.keras.layers.RandomRotation(
            0.4,
            fill_mode="reflect",
            interpolation="bilinear"
        ),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(128, activation="relu"),

        tf.keras.layers.Dense(2, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


eye_model = build_eye_model()
eye_model.summary()

In [ ]:
eye_callback = tf.keras.callbacks.ModelCheckpoint(
    str(EYE_MODEL_PATH),          # Keras 3 는 str 경로 요구
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

In [ ]:
if needs_training(EYE_MODEL_PATH):
    eye_history = eye_model.fit(
        eye_train,
        validation_data=eye_val,
        epochs=EPOCHS,
        callbacks=[eye_callback]
    )
    save_history(eye_history, EYE_HISTORY_PATH)

else:
    print("학습 건너뜀 — 기존 가중치를 사용합니다:", EYE_MODEL_PATH)
    eye_history = load_history(EYE_HISTORY_PATH)

    if eye_history is None:
        print("  (이 모델을 학습할 때의 history 기록이 없어 학습 곡선은 생략됩니다)")

In [ ]:
eye_model = tf.keras.models.load_model(EYE_MODEL_PATH)
print("Eye model 준비 완료 ->", EYE_MODEL_PATH)

## 6. Yawn 모델

In [ ]:
yawn_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=["yawn", "no_yawn"],
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

yawn_val = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=["yawn", "no_yawn"],
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# class 0 = yawn, class 1 = no_yawn
print("\nclass_names:", yawn_train.class_names)

In [ ]:
def build_yawn_model():

    model = tf.keras.Sequential([

        tf.keras.layers.Input(shape=(256, 256, 3)),

        tf.keras.layers.Rescaling(1./255),

        tf.keras.layers.RandomFlip("horizontal"),

        tf.keras.layers.RandomRotation(0.2),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(128, activation="relu"),

        tf.keras.layers.Dense(2, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


yawn_model = build_yawn_model()
yawn_model.summary()

In [ ]:
yawn_callback = tf.keras.callbacks.ModelCheckpoint(
    str(YAWN_MODEL_PATH),
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

In [ ]:
if needs_training(YAWN_MODEL_PATH):
    yawn_history = yawn_model.fit(
        yawn_train,
        validation_data=yawn_val,
        epochs=EPOCHS,
        callbacks=[yawn_callback]
    )
    save_history(yawn_history, YAWN_HISTORY_PATH)

else:
    print("학습 건너뜀 — 기존 가중치를 사용합니다:", YAWN_MODEL_PATH)
    yawn_history = load_history(YAWN_HISTORY_PATH)

    if yawn_history is None:
        print("  (이 모델을 학습할 때의 history 기록이 없어 학습 곡선은 생략됩니다)")

In [ ]:
yawn_model = tf.keras.models.load_model(YAWN_MODEL_PATH)
print("Yawn model 준비 완료 ->", YAWN_MODEL_PATH)

## 7. 학습 곡선

In [ ]:
histories = [(n, h) for n, h in
             [("Eye", eye_history), ("Yawn", yawn_history)] if h is not None]

if not histories:
    print("저장된 history 가 없어 학습 곡선을 그릴 수 없습니다.")
    print("곡선을 보려면 FORCE_RETRAIN = True 로 두고 다시 학습하세요.")

else:
    fig, axes = plt.subplots(len(histories), 2, figsize=(12, 4 * len(histories)))
    axes = np.atleast_2d(axes)

    for row, (name, hist) in enumerate(histories):
        axes[row, 0].plot(hist.history["accuracy"], label="train")
        axes[row, 0].plot(hist.history["val_accuracy"], label="val")
        axes[row, 0].set_title(f"{name} accuracy")
        axes[row, 0].set_xlabel("epoch")
        axes[row, 0].legend()
        axes[row, 0].grid(alpha=0.3)

        axes[row, 1].plot(hist.history["loss"], label="train")
        axes[row, 1].plot(hist.history["val_loss"], label="val")
        axes[row, 1].set_title(f"{name} loss")
        axes[row, 1].set_xlabel("epoch")
        axes[row, 1].legend()
        axes[row, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    for name, hist in histories:
        print(f"{name:5s} best val_accuracy: {max(hist.history['val_accuracy']):.4f}")

## 8. 검증 샘플 확인

In [ ]:
for images, labels in eye_val.take(1):

    pred = eye_model.predict(images[:1], verbose=0)

    pred_class = np.argmax(pred[0])
    true_class = np.argmax(labels[0])

    names = ["Closed", "Open"]

    print("실제:", names[true_class])
    print("예측:", names[pred_class])
    print("확률:", pred[0])

    plt.imshow(images[0].numpy().astype("uint8"))
    plt.axis("off")
    plt.show()

In [ ]:
for images, labels in yawn_val.take(1):

    pred = yawn_model.predict(images[:1], verbose=0)

    pred_class = np.argmax(pred[0])
    true_class = np.argmax(labels[0])

    names = ["Yawn", "No Yawn"]

    print("실제:", names[true_class])
    print("예측:", names[pred_class])
    print("확률:", pred[0])

    plt.imshow(images[0].numpy().astype("uint8"))
    plt.axis("off")
    plt.show()

## 9. 저장 결과

In [ ]:
print("저장된 산출물:", ARTIFACT_DIR)
print()

for p in sorted(ARTIFACT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:24s} {p.stat().st_size/1024/1024:8.2f} MB")

print()
ok = EYE_MODEL_PATH.exists() and YAWN_MODEL_PATH.exists()
print("학습 완료." if ok else "[!] 모델 파일이 없습니다.")
print("이제 02_INFER_YuNet.ipynb 를 실행하세요.")

---

## 알려진 이슈 (원본 코드 유지)

이 노트북은 원본(NITHISHM2410) 학습 코드를 그대로 옮긴 것이라 아래를 고치지 않았습니다.

1. **TEST_DIR 미사용** — `validation_split` 으로 만든 val 을 모델 선택(`ModelCheckpoint`)과
   성능 보고에 모두 써서 낙관 편향이 있습니다. `TEST_DIR` 로 따로 평가해야 합니다.
2. **Yawn 10에폭 미수렴** — loss 가 아직 내려가는 중입니다. 에폭을 늘리고
   `EarlyStopping` 을 붙이는 편이 낫습니다.
3. **`RandomRotation(0.4)`** (Eye) — 최대 ±72°. 눈 crop 에 과하고 Yawn(0.2) 과 다를 이유가 없습니다.
4. **입력 256×256** — 눈 crop 원본이 85~300px 라 오히려 업샘플링입니다.
   128×128 로 줄이면 정확도 손실 없이 추론이 4배 빨라질 여지가 있습니다.